<a href="https://colab.research.google.com/github/Cooper30/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cooper30/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is **Refresh / Content Opportunity Scoring**.

I frame this primarily as a **ranking/scoring problem** because the final decision is not only whether a page is declining, but which pages a content editor should review first. Each page can receive a review-priority score, and the pages with the highest scores can be placed at the top of a refresh queue.

A classification signal such as whether a page is declining can support this ranking, but the operational output I want is an ordered list of content opportunities.

In [10]:
import os
import subprocess
import pandas as pd

REPO_URL = "https://github.com/Cooper30/flyrank-ml-internship.git"
REPO_DIR = "/content/flyrank-ml-internship"

# Repo henüz Colab'a indirilmediyse indir
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

# Repo klasörüne geç
os.chdir(REPO_DIR)

# Dataset yolu
DATA_PATH = "data/raw/content_refresh_anonymized.csv"

print("Current directory:", os.getcwd())
print("Dataset exists:", os.path.exists(DATA_PATH))

# Dataset'i yükle
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

df.head()

Current directory: /content/flyrank-ml-internship
Dataset exists: True
Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

For this starter exercise, I will use **page decline** as a proxy target.

The proxy is:

- **1 = the page is declining**
- **0 = the page is not declining**

In the starter dataset, this proxy is defined from `trend_direction == "down"`.

This is a **rule-derived proxy**, not an independently observed future outcome. Therefore, I should not claim that the model predicts whether refreshing a page will succeed.

Because `is_declining_label` comes from `trend_direction`, I should not use `trend_direction` or `trend_pct` as model features.

In [11]:
df["target_proxy_declining"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

target_counts = df["target_proxy_declining"].value_counts().sort_index()

print("Target counts:")
print(target_counts)

print(
    "\nDeclining rate:",
    round(df["target_proxy_declining"].mean(), 3)
)

Target counts:
target_proxy_declining
0    13738
1    16262
Name: count, dtype: int64

Declining rate: 0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My primary success metric is **Precision@50**.

The goal is to give a content editor a ranked list of pages that should be reviewed first. Precision@50 measures how many of the top 50 recommended pages are actually declining according to the proxy target.

I will consider the prototype useful if it clearly performs better than a simple fixed-rule baseline. As a working target, I will aim for **Precision@50 >= 0.50**, meaning that at least 25 of the top 50 recommended pages match the decline proxy.

In [12]:
import numpy as np
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    top_k_labels = np.asarray(labels)[order[:k]]
    return float(top_k_labels.mean())

print("Primary metric: Precision@50")
print("Working success threshold: >= 0.50")
print("Interpretation: at least 25 of the top 50 pages match the decline proxy.")

Primary metric: Precision@50
Working success threshold: >= 0.50
Interpretation: at least 25 of the top 50 pages match the decline proxy.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is **one content item (one page)**.

Each row represents a single page and contains information about its content characteristics, search performance, engagement, age, and update history.

For the Refresh / Content Opportunity Scoring lane, I will use a subset of these signals to represent each page. The target proxy is shown separately.

`content_id` is included only to identify each row; it would not be used as a predictive feature.

In [13]:
lane_columns = [
    "content_id",
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "word_count",
]

lane_df = df[lane_columns].copy()

lane_df["target_proxy_declining"] = df["target_proxy_declining"]

print("Lane dataframe shape:", lane_df.shape)
print("Unit of analysis: one row = one content item/page")

lane_df.head(10)

Lane dataframe shape: (30000, 11)
Unit of analysis: one row = one content item/page


,content_id,content_type,content_age_days,days_since_last_update,impressions_90d,clicks_90d,avg_position,ctr,engagement_rate,word_count,target_proxy_declining
0,content_304f48230142,keyword article,187,20,3803,29,10.6,0.76,5.88,3221.0,1
1,content_a1fb4e703a9e,keyword article,445,25,15320,7,20.3,0.05,0.00,2481.0,1
2,content_9aa793d4d895,keyword article,141,20,12581,11,36.5,0.09,0.00,3515.0,1
3,content_331d6c4de07b,keyword article,463,22,11751,58,6.2,0.49,1.28,NaN,0
4,content_d99b7a2d90ca,keyword article,263,14,19140,24,44.0,0.13,0.00,2803.0,1
5,content_d4084a4bc775,keyword article,147,20,3970,1,8.5,0.03,0.00,3080.0,1
6,content_9a34b442b552,keyword article,90,20,20,0,7.0,0.00,0.00,3059.0,1
7,content_a63219c6e95a,keyword article,445,22,1724,1,21.2,0.06,3.57,NaN,0
8,content_5e6c160719bc,keyword article,90,20,32574,29,46.0,0.09,5.88,3807.0,1
9,content_c27558df2b0c,keyword article,257,104,1240,2,4.9,0.16,0.00,NaN,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule can be useful as a baseline, but this decision depends on several signals at the same time.

For example, an old page is not automatically a good refresh candidate. A page may be old but still perform well. A highly visible page may deserve attention, but its position, CTR, engagement, content age, update history, and demand can change the decision.

A rule such as:

"review every page older than 180 days with at least 500 impressions"

uses only two thresholds and cannot represent interactions between many signals.

ML can learn combinations of these signals and produce a continuous score that allows pages to be ranked. The fixed rule should still be kept as a transparent baseline, and ML only earns its place if it improves the decision-support ranking.

In [14]:
simple_rule = (
    (df["days_since_last_update"] >= 180) &
    (df["impressions_90d"] >= 500)
)

rule_check = pd.crosstab(
    simple_rule,
    df["target_proxy_declining"],
    rownames=["Fixed rule selected"],
    colnames=["Decline proxy"],
    margins=True
)

display(rule_check)

comparison_features = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate"
]

comparison_table = (
    df.groupby("target_proxy_declining")[comparison_features]
    .median()
    .round(2)
)

display(comparison_table)

Decline proxy,0,1,All
Fixed rule selected,,,
False,13737,16246,29983
True,1,16,17
All,13738,16262,30000


,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,engagement_rate
target_proxy_declining,,,,,,
0,472.0,10.05,0.04,287.0,20.0,0.0
1,961.0,11.30,0.08,216.0,20.0,0.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.